# 🌿 Branching Dialogue Tree MVP — Qwen3-TTS

This notebook demonstrates using Qwen3-TTS VoiceDesign to generate audio for an interactive branching dialogue tree. It builds a graph of dialogue nodes and uses VoiceDesign to create consistent character audio for every node.

Game engines often rely on JSON manifests and split audio files to sequence dialogue during runtime based on player choices. We'll generate exactly that!

In [ ]:
!pip install -q qwen-tts soundfile
import os
import gc
import json
import torch
import numpy as np
import soundfile as sf
from IPython.display import Audio, display
from qwen_tts import Qwen3TTSModel

In [ ]:
OUTPUT_DIR = "/content/dialogue_tree"
os.makedirs(OUTPUT_DIR, exist_ok=True)

NPC = {
    "name": "Elder Varek",
    "voice_prompt": "A tired, wise elder — slow measured speech, weight of decades behind every word, not unkind but not patient either"
}

DIALOGUE_TREE = {
    "root": {
        "node_id": "root",
        "speaker": "NPC",
        "text": "You've come a long way, stranger. The question is — why?",
        "choices": [
            {"label": "seek_help", "player_text": "[I need your help]", "next_node": "help_path"},
            {"label": "demand_info", "player_text": "[I want information]", "next_node": "info_path"},
            {"label": "threaten", "player_text": "[I'm not here to chat]", "next_node": "threat_path"},
            {"label": "leave", "player_text": "[Never mind]", "next_node": "leave_path"}
        ]
    },
    "help_path": {
        "node_id": "help_path",
        "speaker": "NPC",
        "text": "Help. Everyone wants help. Very well. Tell me what you need, and I'll tell you what it'll cost you.",
        "choices": [
            {"label": "explain_quest", "player_text": "[Explain the situation]", "next_node": "help_accept"},
            {"label": "refuse_price", "player_text": "[I can't pay]", "next_node": "help_reject"}
        ]
    },
    "help_accept": {
        "node_id": "help_accept",
        "speaker": "NPC",
        "text": "Hmm. That is... actually worth my time. Here's what you'll need to do. Listen carefully — I won't repeat myself.",
        "choices": []
    },
    "help_reject": {
        "node_id": "help_reject",
        "speaker": "NPC",
        "text": "Then we have nothing further to discuss. Knowledge is not charity, stranger.",
        "choices": []
    },
    "info_path": {
        "node_id": "info_path",
        "speaker": "NPC",
        "text": "Information. That's a dangerous thing to want around here. What exactly are you looking for?",
        "choices": [
            {"label": "ask_about_ruins", "player_text": "[The ruins to the east]", "next_node": "info_ruins"},
            {"label": "ask_about_enemy", "player_text": "[The enemy forces]", "next_node": "info_enemy"}
        ]
    },
    "info_ruins": {
        "node_id": "info_ruins",
        "speaker": "NPC",
        "text": "The eastern ruins. Of course. No one who goes there comes back unchanged — if they come back at all. You want my advice? Don't.",
        "choices": []
    },
    "info_enemy": {
        "node_id": "info_enemy",
        "speaker": "NPC",
        "text": "The forces gathering in the valley. I've been watching them for three weeks. They're not an army. They're a signal. Something is coming behind them.",
        "choices": []
    },
    "threat_path": {
        "node_id": "threat_path",
        "speaker": "NPC",
        "text": "Ha. I've lived through four wars, two plagues, and one very bad harvest. You don't frighten me, child. Put away whatever you're reaching for and try again.",
        "choices": [
            {"label": "back_down", "player_text": "[Stand down]", "next_node": "threat_deescalate"},
            {"label": "attack", "player_text": "[Attack]", "next_node": "threat_fight"}
        ]
    },
    "threat_deescalate": {
        "node_id": "threat_deescalate",
        "speaker": "NPC",
        "text": "Wise. Now. Start over. Properly this time.",
        "choices": []
    },
    "threat_fight": {
        "node_id": "threat_fight",
        "speaker": "NPC",
        "text": "...Really. GUARDS.",
        "choices": []
    },
    "leave_path": {
        "node_id": "leave_path",
        "speaker": "NPC",
        "text": "Safe travels then. The road is less forgiving than I am.",
        "choices": []
    }
}

In [ ]:
# Free up VRAM before loading model
gc.collect()
torch.cuda.empty_cache()

print("Loading Qwen3-TTS VoiceDesign model...")
model_id = "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign"
model = Qwen3TTSModel.from_pretrained(
    model_id, 
    device_map="cuda:0", 
    dtype=torch.bfloat16, 
    attn_implementation="sdpa"
)
print("Model loaded successfully!")

In [ ]:
sample_rate = 24000
generated_audio = {}

print(f"Generating Dialogue Nodes for {NPC['name']}...")
voice_prompt = NPC["voice_prompt"]

for node_id, node_data in DIALOGUE_TREE.items():
    print(f"Processing node: {node_id}")
    audio = model.generate_voice_design(node_data["text"], "English", voice_prompt)
    
    file_name = f"dialogue_{node_id}.wav"
    file_path = f"{OUTPUT_DIR}/{file_name}"
    sf.write(file_path, audio, sample_rate)
    
    generated_audio[node_id] = audio
    print(f"Node: {node_id}")
    display(Audio(file_path))

In [ ]:
def print_tree(node_id, indent=""):
    node = DIALOGUE_TREE[node_id]
    print(f'{indent}{node_id.upper()}: "{node["text"]}"')
    
    for i, choice in enumerate(node.get("choices", [])):
        is_last = (i == len(node["choices"]) - 1)
        prefix = "└── " if is_last else "├── "
        next_indent = indent + ("    " if is_last else "│   ")
        
        print(f'{indent}{prefix}[{choice["label"]}] → {choice["next_node"]}')
        print_tree(choice["next_node"], next_indent)

print("Dialogue Tree Structure:")
print("=" * 50)
print_tree("root")
print("=" * 50)

In [ ]:
# Simulate walking through one full path (root → info_path → info_ruins)
path_nodes = ["root", "info_path", "info_ruins"]
walkthrough_clips = []

for i, node_id in enumerate(path_nodes):
    walkthrough_clips.append(generated_audio[node_id])
    if i < len(path_nodes) - 1:
        # Add 1s silence for player reading/choice time
        walkthrough_clips.append(np.zeros(int(sample_rate * 1.0)))

walkthrough_audio = np.concatenate(walkthrough_clips)
walkthrough_path = f"{OUTPUT_DIR}/dialogue_walkthrough_info_path.wav"
sf.write(walkthrough_path, walkthrough_audio, sample_rate)

print("Walkthrough Simulation (root → info_path → info_ruins):")
display(Audio(walkthrough_path))

*This simulates how a game engine would sequence the dialogue at runtime based on player selections, playing independent audio files separated by the time taken for the player to make a choice.*

In [ ]:
manifest = {}
for node_id, node_data in DIALOGUE_TREE.items():
    manifest[node_id] = {
        "text": node_data["text"],
        "audio_file": f"dialogue_{node_id}.wav",
        "choices": node_data.get("choices", [])
    }

manifest_path = f"{OUTPUT_DIR}/dialogue_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=4)

print("Dialogue Manifest exported for game engine integration:")
print(json.dumps(manifest, indent=2)[:500] + "\n... [truncated] ...\n}")

In [ ]:
import shutil
from google.colab import files

print("Zipping outputs for download...")
shutil.make_archive("/content/dialogue_tree", 'zip', OUTPUT_DIR)
print("Downloading dialogue_tree.zip...")
files.download("/content/dialogue_tree.zip")